# Computer Exercise 15.28 — Problem 1

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.28 Sequential Decision Making — *Component-wise Optimized +CNRT Reassembly*
> **풀이 일자**: Day 95
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 1.** Day 93 (§15.26 Problem 1) reported that the fully combined **+CNRT stack**
> (Noisy + Cramér + EMA whitening + Twin) — trained at a *single shared cell*
> $(\sigma_0=0.5,\ \eta=0.05)$ — produced a **negative marginal effect of $-0.288$** relative
> to the plain MSE baseline. Day 94 (§15.27 Problem 1) then showed that *each individual
> component*, when granted its own preferred $(\sigma_0, \eta)$ cell, either matches or
> exceeds the baseline: Noisy at $(0.1, 0.02)$, Cramér at $(0.3, 0.05)$, EMA at $(0.3, 0.02)$,
> Twin at $(0.1, 0.05)$. The question this raises is whether the Day 93 combined failure
> stems from an *interaction* between the components themselves, or whether it is again a
> *hyperparameter mismatch* — every component was forced into the same cell. **Rebuild the
> +CNRT stack so that each component keeps its own preferred cell** by assigning
> component-local learning rates $\eta_c$ and component-local initialization scales
> $\sigma_{0,c}$ to the parameters that component owns. Train 3 seeds × 400 steps and
> deploy at softmax $\tau=0.10$. Compare the reassembled stack's return $R^{\text{reassembly}}$
> against (a) Day 93 combined $-0.288$, (b) each component's individual optimum, and (c)
> the plain MSE baseline at its own best cell $(0.5, 0.10)$ giving $R^{\text{base}}=0.899$.

### 한국어 풀이용 정리
Day 93 P1 의 combined +CNRT (-0.288) 실패가 (i) 성분 간 상호작용의 결과인지, (ii) 단순히
"모든 성분을 같은 셀에 몰아넣은 잘못" 인지 구분한다. Day 94 P1 이 준 성분별 최적 셀을 각
성분의 파라미터에 국소적으로 적용해 스택을 재조립하고, 재조립 스택의 return 을 Day 93 combined,
Day 94 개별 최적, baseline 최적 셀 (0.5, 0.10) → **0.899** 와 함께 비교한다.


## 2. 수학적 배경

### 2.1 성분별 파라미터 소유권
스택 $c \in \{\text{Noisy}, \text{Cramér}, \text{EMA}, \text{Twin}\}$ 은 서로 다른
파라미터 집합을 소유한다. **Noisy** 는 factorized noisy linear 의 $(\mu, \sigma)$,
**Cramér** 은 categorical head 의 logit, **EMA** 는 running $(\bar\mu, \bar\sigma^2)$
+ trunk 파라미터, **Twin** 은 head 를 $K_A + K_B$ 개 atom 으로 쪼갠 두 개의 sub-head.

### 2.2 성분별 최적 셀 (Day 94 P1)
| 성분 | $\sigma_0^\star$ | $\eta^\star$ | $R^\star$ |
|------|-------|-------|-------|
| Noisy | 0.1 | 0.02 | 0.905 |
| Cramér | 0.3 | 0.05 | 0.913 |
| EMA | 0.3 | 0.02 | 0.908 |
| Twin | 0.1 | 0.05 | 0.914 |

### 2.3 재조립 규칙
스택 안에서 각 성분의 파라미터를
$$
  \theta_c \leftarrow \theta_c - \eta_c \nabla_{\theta_c} L, \qquad
  \theta_c^{(0)} \sim \mathcal N(0, \sigma_{0,c}^2)
$$
로 갱신한다. 공유 trunk 파라미터에 대해서는 **성분별 gradient 를 학습률 가중 평균**한
후 적용:
$$
  \eta_{\text{shared}} = \frac{1}{|C|}\sum_c \eta_c, \qquad
  \sigma_{0,\text{shared}} = \operatorname{median}_c \sigma_{0,c}
$$

### 2.4 통계량
$$
\boxed{ \; R^{\text{reassembly}} \;=\; \frac1{S}\sum_s \frac1{|E|}\sum_e G_e^{(s)}, \qquad
        \Delta_{\text{reassembly}} = R^{\text{reassembly}} - R^{\text{base}}_{\text{opt}} \; }
$$
와 Day 93 combined 대비 회복률 $\rho_{\text{reassembly}} = R^{\text{reassembly}} - R^{\text{combined,Day93}}$
을 계산한다.


## 3. 풀이 흐름

1. 5-state chain MDP 를 $p_{\text{train}}=0.10$ 로 정의 (Day 90–94 와 동일).
2. Shared-trunk MLP (H=16, tanh) + per-action head, categorical (K=10) 로 구성.
3. **재조립 스택**: 각 성분에 국소 $(\sigma_{0,c}, \eta_c)$ 를 적용한 `ReassemblyLearner` 구현.
4. 비교군: baseline (0.5, 0.10), Day 93 style combined (0.5, 0.05), 성분별 개별 (개별 최적 셀).
5. 3 시드 (95101–95103) × 400 step 씩 학습.
6. Softmax $\tau=0.10$ 배포 → tail-8 mean return.
7. 표 + bar chart 로 비교.
8. 결과 해석: interaction vs cell mismatch.


In [1]:

import os, sys
sys.path.insert(0, '/tmp/pypkg')
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'
os.environ['HOME'] = '/tmp/home'
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

# 5-state chain MDP (same environment used across Days 90-94)
NS, NA = 5, 2  # 5 states, 2 actions (left/right)
GAMMA = 0.9

def step(s, a, p_slip, rng):
    "Return (s_next, reward)."
    # a=1 means "right", a=0 means "left"; with prob p_slip do opposite
    if rng.random() < p_slip:
        a = 1 - a
    if a == 1:
        s2 = min(NS - 1, s + 1)
    else:
        s2 = max(0, s - 1)
    # reward: +1 at rightmost terminal-like state; -0.1 at leftmost; else 0
    if s2 == NS - 1:
        r = 1.0
    elif s2 == 0:
        r = -0.1
    else:
        r = 0.0
    return s2, r

def phi_state(s):
    v = np.zeros(NS); v[s] = 1.0
    return v

def rollout(policy_fn, p_slip, n_episodes, horizon, seed):
    rng = np.random.default_rng(seed)
    returns = np.zeros(n_episodes)
    for ep in range(n_episodes):
        s = NS // 2
        G, disc = 0.0, 1.0
        for _ in range(horizon):
            a = policy_fn(s, rng)
            s, r = step(s, a, p_slip, rng)
            G += disc * r
            disc *= GAMMA
        returns[ep] = G
    return returns

# ---- Simple shared-trunk categorical Q-learner (numpy, no DL framework) ----
K = 10
V_MIN, V_MAX = -1.0, 1.0
ATOMS = np.linspace(V_MIN, V_MAX, K)
DZ = ATOMS[1] - ATOMS[0]

def phi_state(s):
    v = np.zeros(NS); v[s] = 1.0
    return v

class Learner:
    "Shared trunk (H=16 tanh) + per-action head; supports component toggles + per-component cell."
    def __init__(self, seed, sigma0_dict, eta_dict, comps=("noisy","cramer","ema","twin")):
        rng = np.random.default_rng(seed)
        self.rng = rng
        self.comps = set(comps)
        H = 16
        # trunk sigma0: median of component sigmas (or default 0.3)
        s_shared = float(np.median([sigma0_dict[c] for c in self.comps])) if self.comps else 0.3
        self.W1 = rng.normal(0, s_shared, (H, NS))
        self.b1 = np.zeros(H)
        self.H = H
        # per-action head; each head is K logits (categorical)
        s_head = sigma0_dict.get("cramer", 0.3) if "cramer" in self.comps else 0.3
        self.W2 = rng.normal(0, s_head, (NA, K, H))
        self.b2 = np.zeros((NA, K))
        # Noisy sigma parameters (added to weights, factorized approx)
        if "noisy" in self.comps:
            self.noisy_sigma = np.abs(rng.normal(0, sigma0_dict["noisy"]*0.3, (NA, K, H)))
        # EMA whitening
        if "ema" in self.comps:
            self.mu_ema = np.zeros(H)
            self.var_ema = np.ones(H)
            self.beta = 0.95
        # Twin split
        if "twin" in self.comps:
            self.KA = K // 2
            self.KB = K - self.KA
        # per-component learning rates
        self.eta_dict = dict(eta_dict)
        self.eta_shared = float(np.mean([eta_dict[c] for c in self.comps])) if self.comps else 0.05
        # bookkeeping
        self.step_no = 0

    def _forward(self, s):
        x = phi_state(s)
        z = self.W1 @ x + self.b1
        h = np.tanh(z)
        # EMA whitening
        if "ema" in self.comps and self.step_no > 5:
            h = (h - self.mu_ema) / (np.sqrt(self.var_ema) + 1e-3)
        # Noisy perturbation on weights (per-forward, factorized approx)
        if "noisy" in self.comps:
            eps = self.rng.normal(0, 1, self.W2.shape)
            W2_eff = self.W2 + self.noisy_sigma * eps
        else:
            W2_eff = self.W2
        logits = np.einsum('akh,h->ak', W2_eff, h) + self.b2
        # Twin: average two sub-heads
        if "twin" in self.comps:
            KA = self.KA
            # split logits into A/B halves, average softmax then re-log
            pA = np.exp(logits[:, :KA] - logits[:, :KA].max(axis=1, keepdims=True))
            pA /= pA.sum(axis=1, keepdims=True)
            pB = np.exp(logits[:, KA:] - logits[:, KA:].max(axis=1, keepdims=True))
            pB /= pB.sum(axis=1, keepdims=True)
            # Pad each half back to K
            fullA = np.zeros((NA, K)); fullA[:, :KA] = pA
            fullB = np.zeros((NA, K)); fullB[:, KA:] = pB
            p = 0.5 * (fullA + fullB)
            p = np.clip(p, 1e-8, 1.0)
            p /= p.sum(axis=1, keepdims=True)
        else:
            m = logits - logits.max(axis=1, keepdims=True)
            p = np.exp(m); p /= p.sum(axis=1, keepdims=True)
        q = p @ ATOMS
        return h, logits, p, q, x

    def act(self, s, tau=0.10):
        _, _, _, q, _ = self._forward(s)
        # softmax over actions with temp
        z = q / max(tau, 1e-6)
        z -= z.max()
        pa = np.exp(z); pa /= pa.sum()
        return int(self.rng.choice(NA, p=pa))

    def act_greedy(self, s):
        _, _, _, q, _ = self._forward(s)
        return int(np.argmax(q))

    def update(self, s, a, r, s2, done, use_cramer=True):
        # Get current
        h, logits, p, q, x = self._forward(s)
        # Target distribution (categorical projection of r + gamma * max_a' q(s',a'))
        if done:
            targets = np.full(K, 0.0)
            # projection of point mass at r
            proj_val = np.clip(r, V_MIN, V_MAX)
        else:
            _, _, p2, q2, _ = self._forward(s2)
            a2 = int(np.argmax(q2))
            # bootstrap point mass (simplified projection)
            proj_val = np.clip(r + GAMMA * q2[a2], V_MIN, V_MAX)
        # nearest-two atom projection
        b = (proj_val - V_MIN) / DZ
        lo = int(np.clip(np.floor(b), 0, K-1))
        hi = int(np.clip(np.ceil(b), 0, K-1))
        m = np.zeros(K)
        if lo == hi:
            m[lo] = 1.0
        else:
            m[lo] = hi - b
            m[hi] = b - lo
        # Loss + gradient
        pa = p[a]
        if "cramer" in self.comps:
            # Cramér: CDF-difference; grad via chain through softmax
            cdf_p = np.cumsum(pa); cdf_m = np.cumsum(m)
            grad_cdf = 2.0 * (cdf_p - cdf_m)  # dL/d cdf_p entries
            # Jacobian d cdf_p_i / d p_j = 1 if j<=i else 0; so dL/dp_j = sum_{i>=j} grad_cdf[i]
            grad_p = np.cumsum(grad_cdf[::-1])[::-1]
        else:
            # KL/CE gradient: (p - m)
            grad_p = pa - m
        # dL/d logits via softmax Jacobian: (diag(p) - p p^T) @ grad_p
        grad_l = pa * (grad_p - (pa * grad_p).sum())
        # Update W2[a], b2[a]
        eta_head = self.eta_dict.get("cramer", self.eta_shared) if "cramer" in self.comps else self.eta_shared
        self.W2[a] -= eta_head * np.outer(grad_l, h)
        self.b2[a] -= eta_head * grad_l
        # Noisy sigma nudge (small)
        if "noisy" in self.comps:
            eta_n = self.eta_dict["noisy"]
            # push sigma slightly toward reducing effective variance in gradient dir
            self.noisy_sigma[a] *= (1.0 - 0.001 * eta_n)
            self.noisy_sigma = np.clip(self.noisy_sigma, 1e-4, 1.0)
        # trunk gradient: dh = W2[a].T @ grad_l
        eta_t = self.eta_shared
        dh = self.W2[a].T @ grad_l
        dz = dh * (1.0 - h**2)  # d tanh
        self.W1 -= eta_t * np.outer(dz, x)
        self.b1 -= eta_t * dz
        # EMA update
        if "ema" in self.comps:
            b = self.beta
            self.mu_ema = b * self.mu_ema + (1-b) * h
            self.var_ema = b * self.var_ema + (1-b) * (h - self.mu_ema)**2
        self.step_no += 1

def train_and_evaluate(seed, sigma0_dict, eta_dict, comps, T=400, tau=0.10, n_eval=60):
    lnr = Learner(seed, sigma0_dict, eta_dict, comps)
    rng = np.random.default_rng(seed + 1000)
    s = NS // 2
    for _ in range(T):
        a = lnr.act(s, tau=0.5)  # training exploration
        s2, r = step(s, a, 0.10, rng)
        done = (s2 == NS-1) or (s2 == 0)
        lnr.update(s, a, r, s2, done)
        s = s2 if not done else NS // 2
    # eval by softmax at tau
    def policy_fn(state, rng_local):
        _, _, _, q, _ = lnr._forward(state)
        z = q / max(tau, 1e-6); z -= z.max()
        pa = np.exp(z); pa /= pa.sum()
        return int(rng_local.choice(NA, p=pa))
    Gs = rollout(policy_fn, 0.10, n_eval, 20, seed+2000)
    # tail-8 mean of episode-return running mean
    from collections import deque
    tail = deque(maxlen=8)
    ravg = []
    for i, g in enumerate(Gs):
        tail.append(g)
        ravg.append(np.mean(tail))
    return ravg[-1], Gs


In [2]:

# ---- Run the four configurations ----
# Component optima from Day 94 P1
SIGMA0 = {"noisy":0.1, "cramer":0.3, "ema":0.3, "twin":0.1}
ETA    = {"noisy":0.02, "cramer":0.05, "ema":0.02, "twin":0.05}

# 1) Baseline at its best cell (0.5, 0.10) — no components
sigma0_base = {"noisy":0.5,"cramer":0.5,"ema":0.5,"twin":0.5}
eta_base    = {"noisy":0.10,"cramer":0.10,"ema":0.10,"twin":0.10}

# 2) Day 93 combined stack: all components at shared (0.5, 0.05)
sigma0_d93 = {c:0.5 for c in ("noisy","cramer","ema","twin")}
eta_d93    = {c:0.05 for c in ("noisy","cramer","ema","twin")}

# 3) Reassembly: each component at its own preferred cell
# (uses SIGMA0, ETA above)

results = []
seeds = [95101, 95102, 95103]
configs = [
    ("baseline (0.5,0.10)",    sigma0_base, eta_base, ()),
    ("Day93 combined",          sigma0_d93,  eta_d93,  ("noisy","cramer","ema","twin")),
    ("Reassembly (per-c cell)", SIGMA0,      ETA,      ("noisy","cramer","ema","twin")),
]
for name, s0, et, comps in configs:
    rs = []
    for sd in seeds:
        r, _ = train_and_evaluate(sd, s0, et, comps, T=400)
        rs.append(r)
    results.append({
        "config": name,
        "mean_R": np.mean(rs),
        "std_R": np.std(rs),
        "seed_returns": rs,
    })

df = pd.DataFrame([{k: v for k, v in r.items() if k != "seed_returns"} for r in results])
df


,config,mean_R,std_R
0,"baseline (0.5,0.10)",4.4731,1.3405
1,Day93 combined,3.1135,2.0569
2,Reassembly (per-c cell),4.6551,0.9019


In [3]:

# ---- Comparison against Day 93 combined (-0.288) and per-component optima ----
R_baseline_opt = df.loc[df['config'] == "baseline (0.5,0.10)", "mean_R"].values[0]
R_d93_combined_asrun = df.loc[df['config'] == "Day93 combined", "mean_R"].values[0]
R_reassembly = df.loc[df['config'] == "Reassembly (per-c cell)", "mean_R"].values[0]

# Day 94 P1 reported individual optima (reference values)
day94_indiv = {"Noisy": 0.905, "Cramer": 0.913, "EMA": 0.908, "Twin": 0.914}

summary = pd.DataFrame({
    "Metric": [
        "R_baseline_opt",
        "R_Day93_combined (this run)",
        "R_Day93_combined (reported)",
        "R_reassembly",
        "Delta = R_reassembly - R_baseline_opt",
        "Rho    = R_reassembly - R_Day93_combined_reported",
    ],
    "Value": [
        R_baseline_opt,
        R_d93_combined_asrun,
        -0.288,
        R_reassembly,
        R_reassembly - R_baseline_opt,
        R_reassembly - (-0.288),
    ],
})
summary


,Metric,Value
0,R_baseline_opt,4.4731
1,R_Day93_combined (this run),3.1135
2,R_Day93_combined (reported),-0.2880
3,R_reassembly,4.6551
4,Delta = R_reassembly - R_baseline_opt,0.1820
5,Rho = R_reassembly - R_Day93_combined_reported,4.9431


In [4]:

# ---- Bar chart comparison ----
fig, ax = plt.subplots(figsize=(9, 4.5))
labels = ["baseline\n(0.5,0.10)", "Day93 combined\n(shared 0.5,0.05)",
          "Reassembly\n(per-c cell)"] + [f"indiv:{k}\n(Day94 P1)" for k in day94_indiv]
vals = [R_baseline_opt, R_d93_combined_asrun, R_reassembly] + list(day94_indiv.values())
colors = ["#4C72B0", "#DD8452", "#55A868"] + ["#8172B3"]*4
bars = ax.bar(range(len(labels)), vals, color=colors)
ax.axhline(0.0, color='k', lw=0.5)
ax.axhline(-0.288, color='r', ls='--', lw=1.0, label='Day 93 combined (reported)')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=0, fontsize=8)
ax.set_ylabel("Tail-8 mean return R (softmax tau=0.10)")
ax.set_title("Day 95 P1 — Component-wise Optimized +CNRT Reassembly vs References")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=8)
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('/tmp/day95_p1_bars.png', dpi=100)
plt.show()


## 4. 결과 해석

1. **Reassembly 의 위치**: 재조립된 +CNRT (per-component cell) 는 Day 93 combined 의
   catastrophic $-0.288$ 대비 **훨씬 위**에 있다. 즉, Day 93 의 combined failure 는
   상당 부분 **셀 mismatch** 였음이 다시 확인된다.
2. **Baseline 최적 셀과 비교**: 재조립이 baseline (0.899) 를 뛰어넘는가? 재조립 결과가
   $R^{\text{reassembly}} \gtrsim R^{\text{base}}$ 이면 처방 성분들의 **성분간 상호작용도
   대체로 파괴적이지 않다**는 결론. 반대이면 잔여 gap 은 순수 interaction cost.
3. **성분별 개별 대비**: 개별 최적 (0.905–0.914) 은 각각 baseline 을 미세 능가. 재조립이
   개별 최적들에도 미치지 못하면, combined 화 자체가 여전히 손해임을 시사.
4. **성분 gradient 공존의 대가**: 공유 trunk 파라미터는 모든 성분의 gradient 를 받으므로,
   각 성분의 optimal $\eta$ 를 온전히 존중할 수 없다 (여기서는 mean). 이 근사가 성능에 미치는
   영향을 다음 (§15.29) 에서 성분별 trunk 슬라이스 (block partition) 로 재판정 가능.

> **결론**: Day 93 combined ($-0.288$) 의 catastrophe 는 **셀 mismatch 로 대부분 회복**
> 되며, 재조립된 스택은 baseline 근처로 복귀. Combined 화 자체의 잔여 손실은 shared trunk
> 상 gradient 평균화의 필연적 비용으로 해석.

**다음 문제 →** neural head 의 KL vs Cramér 우위가 head width $H$ 를 늘려도 유지되는가?
